# Diagnose and Fix FastAPI Startup

This notebook will locate the FastAPI entry point, fix startup errors, install missing dependencies, and launch the app on `http://127.0.0.1:8000`.

## 1. Locate the FastAPI entry point

Search the project files for the FastAPI app instance and the script or module used to start uvicorn.

In [ ]:
import os
from pathlib import Path
base = Path(r'd:/Security Advisory Deposit(infinite)/security_advisory_digest')
print('Workspace root:', base)
print('Files:', sorted([p.name for p in base.iterdir() if p.is_file()]))

## 2. Inspect and fix startup errors

Compile the entrypoint and inspect invalid dependencies or misconfigured startup settings.

In [ ]:
import py_compile
try:
    py_compile.compile(base / 'app.py', doraise=True)
    print('app.py compiled successfully')
except Exception as e:
    print('app.py compile failure:', e)

req_file = base / 'requirements.txt'
print('\nrequirements.txt contents:')
print(req_file.read_text(encoding='utf-8'))

text = req_file.read_text(encoding='utf-8')
if 'sqlite3-python' in text:
    fixed = '\n'.join(line for line in text.splitlines() if 'sqlite3-python' not in line)
    req_file.write_text(fixed, encoding='utf-8')
    print('\nRemoved invalid sqlite3-python dependency from requirements.txt')
    print(req_file.read_text(encoding='utf-8'))
else:
    print('\nNo invalid sqlite3-python dependency found')


## 3. Install missing dependencies

Use pip to install libraries required by FastAPI, uvicorn, and the app dependencies.

In [ ]:
import sys, subprocess
os.chdir(base)
print('Installing dependencies from requirements.txt...')
res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], capture_output=True, text=True)
print('returncode', res.returncode)
print('stdout', res.stdout[:2000])
print('stderr', res.stderr[:2000])

## 4. Run the FastAPI app

Start the application with uvicorn on the configured host and port.

In [ ]:
import subprocess, time
os.chdir(base)
log_path = base / 'uvicorn_start.log'
with open(log_path, 'a', encoding='utf-8') as log_file:
    proc = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', '8000'], stdout=log_file, stderr=subprocess.STDOUT)
print('Started uvicorn PID', proc.pid)
print('Logs are being written to', log_path)
time.sleep(3)
import socket
s = socket.socket()
s.settimeout(2)
result = s.connect_ex(('127.0.0.1', 8000))
s.close()
print('connect_ex result:', result)
if result == 0:
    print('Server appears to be running at http://127.0.0.1:8000')
else:
    print('Server is not reachable yet; inspect uvicorn_start.log')

## 5. Verify the running documentation URL

If the server is running, open the Swagger UI at `http://127.0.0.1:8000/docs` and confirm it responds.